# Imports

In [17]:
# Linear algebra
import numpy as np

# Dataframes
import pandas as pd

# Path
import os
data_path = os.path.join('..', 'data')

# Load data

In [18]:
c = 0.5

# --- load genes data ---
genes_df = pd.read_pickle(os.path.join(data_path, 'genes_human_discrete_expression.pkl'))

# --- load cluster data ---

# path to cluster file
cluster_path = os.path.join(data_path, f"proteins_human_{int(100 * c)}.fasta.clstr")

# read cluster file
column_names = [0, 1, 2, 3, 4, 5, 6, 7]
cluster_df = pd.read_csv(cluster_path, delimiter=r'\t| |,|>|\|', header=None, engine='python', names=column_names)

# Create clustered dataframe

In [19]:
# --- reorganize cluster data ---

# drop unnecessary columns
cluster_df = cluster_df.drop(columns=[3, 6])

# Fill in cluster numbers for each row
cluster_df[2] = cluster_df[2].ffill()

# drop rows with empty values in column 0
cluster_df = cluster_df[cluster_df[0].notna()]

# replace empty values in column 7 with *
cluster_df[7] = cluster_df[7].fillna('*')

# make column 2 the first column
cluster_df = cluster_df[[2, 0, 1, 4, 5, 7]]

# remove last two characters (letters) from values in column 1
cluster_df[1] = cluster_df[1].str[:-2]

# make column 0, 2 integer valued
cluster_df[0] = cluster_df[0].astype(int)
cluster_df[2] = cluster_df[2].astype(int)

# rename columns
cluster_df.columns = ['cluster', 'sub-idx', 'length (aa)', 'gene', 'transcript', 'identity']

# remove "..." from transcripts
cluster_df['transcript'] = cluster_df['transcript'].str[:-3]

# view cluster dataframe
cluster_df

,cluster,sub-idx,length (aa),gene,transcript,identity
1,0,0,12,ENSG00000115211,ENST00000418146,50.00%
2,0,1,8923,ENSG00000154358,ENST00000570156,100.00%
3,0,2,8925,ENSG00000154358,ENST00000680850,*
4,0,3,6620,ENSG00000154358,ENST00000284548,54.88%
5,0,4,7968,ENSG00000154358,ENST00000422127,66.77%
...,...,...,...,...,...,...
97336,17247,0,28,ENSG00000089335,ENST00000509528,*
97338,17248,0,28,ENSG00000151923,ENST00000479729,*
97340,17249,0,26,ENSG00000284895,ENST00000646142,*
97342,17250,0,25,ENSG00000197128,ENST00000601768,*


## Clean clustered dataframe and merge clusters

In [20]:
# --- Drop 'identity', 'sub-idx' columns ---
cluster_df.drop(columns=['identity', 'sub-idx'], inplace=True)

# --- Place all same-gene rows in the same cluster ---
# Find the minimum cluster value for each gene
min_cluster_per_gene = cluster_df.groupby('gene')['cluster'].transform('min')

# Update the cluster values in the dataframe
cluster_df['cluster'] = min_cluster_per_gene

# Get unique cluster values and sort them
unique_clusters = sorted(cluster_df['cluster'].unique())

# Create a mapping from old cluster values to new continuous integers
cluster_mapping = {old_cluster: new_cluster for new_cluster, old_cluster in enumerate(unique_clusters)}

# Replace old cluster values with new ones
cluster_df['cluster'] = cluster_df['cluster'].map(cluster_mapping)

# Order rows by increasing cluster value
cluster_df.sort_values(by='cluster', inplace=True)
cluster_df

,cluster,length (aa),gene,transcript
7582,0,35,ENSG00000204482,ENST00000376092
92985,0,73,ENSG00000204482,ENST00000376086
92986,0,59,ENSG00000204482,ENST00000303757
92987,0,104,ENSG00000204482,ENST00000376093
92980,0,66,ENSG00000204482,ENST00000418507
...,...,...,...,...
97330,14499,29,ENSG00000288708,ENST00000683730
97332,14500,29,ENSG00000291303,ENST00000706628
97334,14501,29,ENSG00000291314,ENST00000706950
97340,14502,26,ENSG00000284895,ENST00000646142


## Merge with genes data

In [21]:
merged_df = pd.merge(cluster_df, genes_df[['transcript', 'seq', 'amino_acid_seq', 'median']], on='transcript', how='left')
merged_df

,cluster,length (aa),gene,transcript,seq,amino_acid_seq,median
0,0,35,ENSG00000204482,ENST00000376092,ATGTTATCGCGGAATGATGCACCTTCTGTCCTGGTCCCAGGCCCAG...,MLSRNDAPSVLVPGPGLLRAGTPLCISAEAASAQQ*,"(expr_low25, expr_low25, expr_low25, expr_low2..."
1,0,73,ENSG00000204482,ENST00000376086,ATGTTATCGCGGAATGATGTAAAGAGGCTGGAGAGGAGCTGGCACC...,MLSRNDVKRLERSWHLLSWSQAQGSSEQELHYASLQRLPVPSSEGP...,"(expr_low25, expr_low25, expr_low25, expr_low2..."
2,0,59,ENSG00000204482,ENST00000303757,ATGTTATCGCGGAATGATGATATATGTATCTACGGGGGCCTGGGGC...,MLSRNDDICIYGGLGLGGLLLLAVVLLSACLCWLHRRGPGLLRAGT...,"(expr_pre50_75, expr_pre50_75, expr_pre50_75, ..."
3,0,104,ENSG00000204482,ENST00000376093,ATGTTATCGCGGAATGATGATATATGTATCTACGGGGGCCTGGGGC...,MLSRNDDICIYGGLGLGGLLLLAVVLLSACLCWLHRRVKRLERSWH...,"(expr_pre25_50, expr_pre50_75, expr_pre50_75, ..."
4,0,66,ENSG00000204482,ENST00000418507,ATGTTATCGCGGAATGATGTAAAGAGGCTGGAGAGGAGCTGGGCCC...,MLSRNDVKRLERSWAQGSSEQELHYASLQRLPVPSSEGPDLRGRDK...,"(expr_pre25_50, expr_pre50_75, expr_low25, exp..."
...,...,...,...,...,...,...,...
80088,14499,29,ENSG00000288708,ENST00000683730,ATGATTCCTCATGAAGAGCCTGGATCCCCTACAGAAATCAAATGTG...,MIPHEEPGSPTEIKCDFPFIRLKSEPSRQ*,NaN
80089,14500,29,ENSG00000291303,ENST00000706628,ATGACATTTCATCGCAATGTCCGATCGTTTGGGGCAAATTACCAAG...,MTFHRNVRSFGANYQGQGWEKQVLDSQPV*,NaN
80090,14501,29,ENSG00000291314,ENST00000706950,ATGTTTCTTCAAAAAGAACTAGTGTGCAGTCCATTGATAGCTGATC...,MFLQKELVCSPLIADQLPWVLLMTQESFA*,NaN
80091,14502,26,ENSG00000284895,ENST00000646142,ATGAATGCTGGCTTTCAGAGAGAACAGCGTTTCAGTTTTGGTCATC...,MNAGFQREQRFSFGHRKWCLQHRRRA*,NaN


# Separate Data by Clusters

## Shuffle clusters

In [22]:
# Extract unique clusters
unique_clusters = merged_df['cluster'].unique()

# Shuffle the unique clusters
np.random.shuffle(unique_clusters)

# Create a new DataFrame by concatenating the clusters in the shuffled order
shuffled_df = pd.concat([merged_df[merged_df['cluster'] == cluster] for cluster in unique_clusters])

# Reset index if needed
shuffled_df.reset_index(drop=True, inplace=True)
shuffled_df

,cluster,length (aa),gene,transcript,seq,amino_acid_seq,median
0,4973,450,ENSG00000135063,ENST00000455972,ATGATACTCCTGGTAAACCTCTTTGTGCTGCTCTCTGTGGTTTGTG...,MILLVNLFVLLSVVCVLLNLAGFILGCQGAQFVSSVPRCDLVDLGE...,"(expr_pre75_90, expr_pre75_90, expr_pre75_90, ..."
1,4973,450,ENSG00000135063,ENST00000257515,ATGATACTCCTGGTAAACCTCTTTGTGCTGCTCTCTGTGGTTTGTG...,MILLVNLFVLLSVVCVLLNLAGFILGCQGAQFVSSVPRCDLVDLGE...,"(expr_low25, expr_low25, expr_low25, expr_low2..."
2,4973,603,ENSG00000135063,ENST00000303068,ATGTCCCTGCCCGTGGTGCTCCCGGGCTCCTGCTGCCCCGTGGCTG...,MSLPVVLPGSCCPVAGLSGGPQAGGPGAAAAAAQEPPLPPLRPRWP...,"(expr_low25, expr_low25, expr_low25, expr_low2..."
3,4973,151,ENSG00000135063,ENST00000377216,ATGATACTCCTGGTAAACCTCTTTGTGCTGCTCTCTGTGGTTTGTG...,MILLVNLFVLLSVVCVLLNLAGFILGCQGAQFVSSVPRCDLVDLGE...,"(expr_pre75_90, expr_pre75_90, expr_pre50_75, ..."
4,9590,266,ENSG00000133317,ENST00000425950,ATGGTCATGCTGCAAGGAGTGGTCCCTCTAGATGCACACAGGTTTC...,MVMLQGVVPLDAHRFQVDFQCGCSLCPRPDIAFHFNPRFHTTKPHV...,"(expr_low25, expr_low25, expr_low25, expr_low2..."
...,...,...,...,...,...,...,...
80088,11934,228,ENSG00000165233,ENST00000466409,ATGACAGATCAGACCTATTGTGACCGCCTGGTGCAGGACACGCCTT...,MTDQTYCDRLVQDTPFLTGHGRLSEQQVDRIILQLNRYYPQILTNK...,"(expr_pre75_90, expr_pre50_75, expr_pre75_90, ..."
80089,11934,183,ENSG00000165233,ENST00000375464,ATGACAGATCAGACCTATTGTGACCGCCTGGTGCAGGACACGCCTT...,MTDQTYCDRLVQDTPFLTGHGRLSEQQVDRIILQLNRYYPQILTNK...,"(expr_pre75_90, expr_top10, expr_top10, expr_t..."
80090,13931,117,ENSG00000253250,ENST00000517562,ATGGAAACCAAAAAATTAATTGGTAAACCGCTTCAACCAGCAAGAC...,METKKLIGKPLQPARPVRHLTSPPGAVFPFNFQNEYPCNTQCIQSG...,"(expr_pre75_90, expr_pre75_90, expr_pre75_90, ..."
80091,5074,594,ENSG00000144671,ENST00000273173,ATGGCAGGAGAGGAGAACTTCAAGGAAGAGCTCAGATCCCAGGATG...,MAGEENFKEELRSQDASRNLNQHEVAGHPHSWSLEMLLRRLRAVHT...,"(expr_pre25_50, expr_pre25_50, expr_pre25_50, ..."


## Speparate to train, validation, test

In [23]:
def separate_by_clusters(shuffled_dataframe, cluster_column='cluster', train_validation_test=[0.7, 0.1, 0.2]):
    # Take the first (~70%) rows as train set
    num_rows_train = int(train_validation_test[0] * len(shuffled_dataframe))
    last_cluster_train = shuffled_dataframe.iloc[num_rows_train - 1][cluster_column]
    boundary_index_train = shuffled_dataframe[shuffled_dataframe[cluster_column] == last_cluster_train].index[-1] + 1
    train_dataframe = shuffled_dataframe.iloc[0:boundary_index_train]

    # Take the next (~10%) rows as validation set
    num_rows_10_percent = int(train_validation_test[1] * len(shuffled_dataframe))
    last_cluster_validation = shuffled_dataframe.iloc[boundary_index_train + num_rows_10_percent - 1][cluster_column]
    boundary_index_validation = shuffled_dataframe[shuffled_dataframe[cluster_column] == last_cluster_validation].index[-1] + 1
    validation_dataframe = shuffled_dataframe.iloc[boundary_index_train:boundary_index_validation]

    # Take the last (~20%) rows as test set
    test_dataframe = shuffled_dataframe.iloc[boundary_index_validation:]
    return train_dataframe, validation_dataframe, test_dataframe

In [24]:
train_df, validation_df, test_df = separate_by_clusters(shuffled_df)

num_rows = len(shuffled_df)
print(f"Split ratio: [{len(train_df) / num_rows : .5f}, \
      {len(validation_df) / num_rows : .5f}, \
      {(len(test_df) / num_rows) : .5f}]")

Split ratio: [ 0.70010,        0.10043,        0.19947]


In [25]:
train_df

,cluster,length (aa),gene,transcript,seq,amino_acid_seq,median
0,4973,450,ENSG00000135063,ENST00000455972,ATGATACTCCTGGTAAACCTCTTTGTGCTGCTCTCTGTGGTTTGTG...,MILLVNLFVLLSVVCVLLNLAGFILGCQGAQFVSSVPRCDLVDLGE...,"(expr_pre75_90, expr_pre75_90, expr_pre75_90, ..."
1,4973,450,ENSG00000135063,ENST00000257515,ATGATACTCCTGGTAAACCTCTTTGTGCTGCTCTCTGTGGTTTGTG...,MILLVNLFVLLSVVCVLLNLAGFILGCQGAQFVSSVPRCDLVDLGE...,"(expr_low25, expr_low25, expr_low25, expr_low2..."
2,4973,603,ENSG00000135063,ENST00000303068,ATGTCCCTGCCCGTGGTGCTCCCGGGCTCCTGCTGCCCCGTGGCTG...,MSLPVVLPGSCCPVAGLSGGPQAGGPGAAAAAAQEPPLPPLRPRWP...,"(expr_low25, expr_low25, expr_low25, expr_low2..."
3,4973,151,ENSG00000135063,ENST00000377216,ATGATACTCCTGGTAAACCTCTTTGTGCTGCTCTCTGTGGTTTGTG...,MILLVNLFVLLSVVCVLLNLAGFILGCQGAQFVSSVPRCDLVDLGE...,"(expr_pre75_90, expr_pre75_90, expr_pre50_75, ..."
4,9590,266,ENSG00000133317,ENST00000425950,ATGGTCATGCTGCAAGGAGTGGTCCCTCTAGATGCACACAGGTTTC...,MVMLQGVVPLDAHRFQVDFQCGCSLCPRPDIAFHFNPRFHTTKPHV...,"(expr_low25, expr_low25, expr_low25, expr_low2..."
...,...,...,...,...,...,...,...
56068,6796,172,ENSG00000142046,ENST00000392002,ATGGACAGCCCTAGTCTTCGTGAGCTTCAACAGCCTCTGCTGGAGG...,MDSPSLRELQQPLLEGTECETPAQKPGRHELGSPLREIAFAESLRG...,"(expr_pre75_90, expr_pre75_90, expr_pre50_75, ..."
56069,6796,139,ENSG00000142046,ENST00000356385,ATGGACAGCCCTAGTCTTCGTGAGCTTCAACAGCCTCTGCTGGAGG...,MDSPSLRELQQPLLEGTECETPAQKPGRHELGSPLREIAFAESLRG...,"(expr_pre50_75, expr_pre50_75, expr_pre50_75, ..."
56070,6796,133,ENSG00000142046,ENST00000544232,ATGGACAGCCCTAGTCTTCGTGAGCTTCAACAGCCTCTGCTGGAGG...,MDSPSLRELQQPLLEGTECETPAQKPGRHELGSPLREIAFAESLRG...,"(expr_pre50_75, expr_pre75_90, expr_low25, exp..."
56071,6796,190,ENSG00000142046,ENST00000604123,ATGCGCTGTGGCTTTGCGGGCGGTGTGGGTCACCAGAGAAAGAGGA...,MRCGFAGGVGHQRKRTRRRRLNPGDETQGSRPEEGDPERKESQAGK...,"(expr_pre50_75, expr_pre50_75, expr_low25, exp..."


## Shuffle Separated Dataframes

In [26]:
shuffled_train = train_df.sample(frac=1).reset_index(drop=True)
shuffled_validation = validation_df.sample(frac=1).reset_index(drop=True)
shuffled_test = test_df.sample(frac=1).reset_index(drop=True)

# Save Dataframes

In [27]:
shuffled_train.to_pickle(os.path.join(data_path, f"0_train_{int(100 * c)}.pkl"))
shuffled_validation.to_pickle(os.path.join(data_path, f"1_validation_{int(100 * c)}.pkl"))
shuffled_test.to_pickle(os.path.join(data_path, f"2_test_{int(100 * c)}.pkl"))